# Preconditioning

In [ ]:
#    APM41012EP course notebook - Chapter 5 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Iterative methods for solving linear systems
#    Preconditioning
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import time
import numpy as np
from scipy.sparse import diags
from scipy.sparse.linalg import norm

## Poisson equation

We want to solve the elliptic problem given by the Poisson equation subject to Dirichlet boundary conditions:

$$
\left\{
\begin{aligned}
-\Delta u(x) & =  b(x) \quad \text{ in } \; \Omega = [0,1] \quad \text{with} \; b(x)=1\\
        u(0) & =  0 \quad \text{ on }  \;  \partial \Omega
\end{aligned}
\right.
$$

## Chebyshev method

The Chebyshev iterative method also makes it possible to converge towards a solution of the problem.

In [ ]:
def chebychev(a, b, lambda1, lambda2, max_iter, eps=1.e-6, output=False):

    theta = (lambda2 + lambda1) / 2.
    delta = (lambda2 - lambda1) / 2.

    xk = np.zeros(b.size)
    norm_b = np.linalg.norm(b)

    rk = b - a.dot(xk)
    sigma = theta/delta
    rhok = 1./sigma
    dk = (1./theta)*rk

    for k in range(max_iter):
        xk = xk + dk
        rk = rk - a.dot(dk)
        norm_rk = np.linalg.norm(rk)
        if norm_rk/norm_b < eps: break
        rhokm1 = rhok
        rhok = 1./(2.*sigma-rhok)
        dk = rhok*rhokm1*dk + ((2*rhok)/delta)*rk

    if output:
        print(f"  Number of iterations = {k+1}")
        print(f"  ||rk|| / ||b||                      = {norm_rk/norm_b}")

    return xk

### 1d case

In [ ]:
nx = 100
dx = 1/(nx+1)
diagonals = [np.repeat(2/(dx*dx), nx), np.repeat(-1/(dx*dx), nx-1), np.repeat(-1/(dx*dx), nx-1)]
a = diags(diagonals, [0, -1, 1])

b = np.ones(nx)

print("\nSolution using the Chebyshev method")
cst = (4/(dx**2))
# The whole interval of eigenvalues is used
# cf. lecture notes

lambda1 = cst*np.sin((np.pi)/(2*(nx+1)))**2
lambda2 = cst*np.sin((np.pi*nx)/(2*(nx+1)))**2
u = chebychev(a, b, lambda1, lambda2, max_iter=10000, output=True)

res = np.linalg.norm(b - a.dot(u))
norm_a = norm(a)
print(f"  ||A.xk - b|| / ||A|| ||xk|| + ||b|| = {res / (norm_a * np.linalg.norm(u) + np.linalg.norm(b))}")

## Preconditioned conjugate gradient method

In [ ]:
def conjugate_gradient(a, b, eps=1.e-6):
    xk = np.zeros(b.size)
    norm_b = np.linalg.norm(b)

    rk = b - a.dot(xk)
    pk = rk
    rkm1 = rk

    for k in range(b.size):
        apk = a.dot(pk)
        alpha = np.dot(rk,rk) / np.dot(pk, apk)
        xk = xk + alpha*pk
        rk = rk - alpha*apk
        norm_rk = np.linalg.norm(rk)
        #print(k, np.linalg.norm(rk))
        if norm_rk/norm_b < eps: break
        beta = np.dot(rk,rk) / np.dot(rkm1, rkm1)
        pk = rk + beta*pk
        rkm1 = rk

    print(f"  Number of iterations = {k+1}")
    print(f"  ||rk|| / ||b||                      = {norm_rk/norm_b}")

    return xk

def prec_cheb_conjugate_gradient(a, b, lambda1, lambda2, eps=1.e-6, iter_prec=10):

    xk = np.zeros(b.size)
    norm_b = np.linalg.norm(b)
    rk = b - a.dot(xk)
    rkm1 = rk
    zk = chebychev(a, rk, lambda1, lambda2, max_iter=iter_prec)
    zkm1 = zk
    pk = zk

    for k in range(b.size):
        apk = a.dot(pk)
        alpha = np.dot(rk,zk) / np.dot(pk, apk)
        xk = xk + alpha*pk
        rk = rk - alpha*apk
        norm_rk = np.linalg.norm(rk)
        #print('prec', k, np.linalg.norm(rk))
        if norm_rk/norm_b < eps: break
        zk = chebychev(a, rk, lambda1, lambda2, max_iter=iter_prec)
        beta = np.dot(rk,zk) / np.dot(rkm1, zkm1)
        pk = zk + beta*pk
        rkm1 = rk
        zkm1 = zk

    print(f"  Number of iterations = {k+1}")
    print(f"  ||rk|| / ||b||                      = {norm_rk/norm_b}")

    return xk

### 1d case

In [ ]:
nx = 10000
dx = 1/(nx+1)
diagonals = [np.repeat(2/(dx*dx), nx), np.repeat(-1/(dx*dx), nx-1), np.repeat(-1/(dx*dx), nx-1)]
a = diags(diagonals, [0, -1, 1])

b = np.ones(nx)

print("\nSolution using the conjugate gradient method")
u = conjugate_gradient(a, b)
res = np.linalg.norm(b - a.dot(u))
norm_a = norm(a)
print(f"  ||A.xk - b|| / ||A|| ||xk|| + ||b|| = {res / (norm_a * np.linalg.norm(u) + np.linalg.norm(b))}")

print("\nSolution using the conjugate gradient method preconditioned with Chebyshev")
cst = (4/(dx**2))
# heuristic estimate of a "good" interval of eigenvalues to consider - cf. lecture notes
# This estimate is only valid for preconditioning! 
i = 70
alpha = cst*np.sin((np.pi*i)/(2*(nx+1)))**2
beta = cst*np.sin((np.pi*(nx-(i-1)))/(2*(nx+1)))**2
u = prec_cheb_conjugate_gradient(a, b, alpha, beta, iter_prec=10)
res = np.linalg.norm(b - a.dot(u))
print(f"  ||A.xk - b|| / ||A|| ||xk|| + ||b|| = {res / (norm_a * np.linalg.norm(u) + np.linalg.norm(b))}")

### 2d case

In [ ]:
nx = 600
ny = nx
dx = 1/(nx+1)
dy = 1/(ny+1)

# building the sparse matrix
diag = np.repeat(2/dx**2 + 2/dy**2, nx*ny)
diag_x = np.tile(np.repeat([-1/dx**2, 0.], (nx-1, 1)), ny)
diag_y = np.repeat(-1/dy**2, nx*(ny-1))
a = diags([diag, diag_x, diag_x, diag_y, diag_y], [0, -1, 1, -nx, nx])

# right-hand side
b = np.ones(nx*ny)

print(f"2d case: nx = {nx} and ny = {ny} => nx . ny = {nx*ny}")

print("\nSolution using the conjugate gradient method")
u = conjugate_gradient(a, b)
res = np.linalg.norm(b - a.dot(u))
norm_a = norm(a)
print(f"  ||A.xk - b|| / ||A|| ||xk|| + ||b|| = {res / (norm_a * np.linalg.norm(u) + np.linalg.norm(b))}")

print("\nSolution using the conjugate gradient method preconditioned with Chebyshev")
cst = (8/(dx**2))
# heuristic estimate of a "good" interval of eigenvalues to consider - cf. lecture notes
# This estimate is only valid for preconditioning! 
i = 5
alpha = cst*np.sin((np.pi*i)/(2*(nx+1)))**2
beta = cst*np.sin((np.pi*(nx-(i-1)))/(2*(nx+1)))**2
u = prec_cheb_conjugate_gradient(a, b, alpha, beta, iter_prec=10)
res = np.linalg.norm(b - a.dot(u))
print(f"  ||A.xk - b|| / ||A|| ||xk|| + ||b|| = {res / (norm_a * np.linalg.norm(u) + np.linalg.norm(b))}")

### 3d case

In [ ]:
nx = 100
ny = nx
nz = nx
dx = 1/(nx+1)
dy = 1/(ny+1)
dz = 1/(nz+1)

# building the sparse matrix
diag = np.repeat(2/dx**2 + 2/dx**2 + 2/dz**2, nx*ny*nz)
diag_x = np.tile(np.repeat([-1/dx**2, 0.], (nx-1, 1)), ny*nz)
diag_y = np.tile(np.repeat([-1/dy**2, 0.], (nx*(ny-1), nz)), nz)
diag_z = np.repeat(-1/dz**2, nx*ny*(nz-1))
a = diags([diag, diag_x, diag_x, diag_y, diag_y, diag_z, diag_z], [0, -1, 1, -nx, nx, -nx*ny, nx*ny])

# right-hand side
b = np.ones(nx*ny*nz)

print(f"3d case: nx = {nx}, ny = {ny} and nz = {nz} => nx . ny . nz = {nx*ny*nz}")
print("\nSolution using the conjugate gradient method")
u = conjugate_gradient(a, b)
res = np.linalg.norm(b - a.dot(u))
norm_a = norm(a)
print(f"  ||A.xk - b|| / ||A|| ||xk|| + ||b|| = {res / (norm_a * np.linalg.norm(u) + np.linalg.norm(b))}")

print("\nSolution using the conjugate gradient method preconditioned with Chebyshev")
cst = (12/(dx**2))
# heuristic estimate of a "good" interval of eigenvalues to consider - cf. lecture notes
# This estimate is only valid for preconditioning! 
i = 4
alpha = cst*np.sin((np.pi*i)/(2*(nx+1)))**2
beta = cst*np.sin((np.pi*(nx-(i-1)))/(2*(nx+1)))**2
u = prec_cheb_conjugate_gradient(a, b, alpha, beta, iter_prec=10)
res = np.linalg.norm(b - a.dot(u))
print(f"  ||A.xk - b|| / ||A|| ||xk|| + ||b|| = {res / (norm_a * np.linalg.norm(u) + np.linalg.norm(b))}")